<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">模块 11：BI 与 AI 应用</div><div style="color:#17212b;font-size:30px;font-weight:750">模块 11：BI 与 AI 应用</div><p style="color:#475569;line-height:1.7">通过稳定的 SQL 契约向 BI 和 AI 消费者交付经过评审的指标。请按顺序运行；结果会以表格展示，写入只作用于本模块的 `_l2` 对象。</p></div>

## 边界

本实验不会修改 Level 1 源表，只创建或替换带 `_l2` 后缀的对象。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP VIEW IF EXISTS bi_order_metrics_l2")
lab.execute("""
CREATE VIEW bi_order_metrics_l2 AS
SELECT order_date,
       SUM(order_count) AS order_count,
       SUM(gross_amount) AS gross_amount,
       CASE WHEN SUM(order_count) = 0 THEN NULL
            ELSE SUM(gross_amount) / SUM(order_count) END AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date
""")
lab.sql("SELECT * FROM bi_order_metrics_l2 ORDER BY order_date", title="BI 语义视图")

In [ ]:
lab.sql("""
SELECT order_date,
       order_count AS feature_order_count,
       gross_amount AS feature_gross_amount,
       average_order_amount AS feature_average_amount,
       CASE WHEN order_count = 0 THEN 1 ELSE 0 END AS zero_order_flag
FROM bi_order_metrics_l2
ORDER BY order_date
""", title="经过评审的特征投影")

In [ ]:
lab.sql("""
SELECT
    COUNT(*) AS serving_days,
    SUM(CASE WHEN order_count IS NULL THEN 1 ELSE 0 END) AS null_order_days,
    MIN(order_date) AS first_date,
    MAX(order_date) AS last_date,
    SUM(order_count) AS served_orders
FROM bi_order_metrics_l2
""", title="消费者契约检查")

# BI 看板或 AI 应用通过各自的连接器消费此结果。
# Doris 提供经过评审的行，但不会默默定义模型准确性。

## 要点

将结果与课程中声明的业务粒度对照。SQL 执行成功本身不能证明模型、指标、访问边界或消费者契约正确。